# Area Effects

This notebook shows how to:
1. Find items with area/radius effects
2. Extract effect radius information
3. Work with AreaBuff templates
4. Understand different effect scopes

In [9]:
from assetextractor.extraction.utils import Config
from assetextractor.parsing.core.assets import AssetCache
from assetextractor.parsing.core.texts import StandardTextConverter

# Load assets
config = Config.from_json("config.json")
assets = AssetCache.load(config)
texts = assets.texts

# Set language
LANGUAGE = "english"
texts.converter = StandardTextConverter(LANGUAGE)

print("Assets loaded!")

Assets loaded!


## Understanding Effect Scopes

Items have an `EffectScope` property that determines how they apply:

In [10]:
# Get an item and check its scope
item = assets.get(80510)  # Example item

name = item.text()
print(f"Item: {name}\n")

if hasattr(item, 'Effect'):
    scope = item.Effect.EffectScope()
    print(f"Effect Scope (literal): {scope}")
    
    # Get localized scope text using ui_text
    scope_attr = item.Effect.EffectScope
    if hasattr(scope_attr, 'ui_text') and scope_attr.ui_text:
        scope_text = scope_attr.ui_text()
        print(f"Localized: {scope_text}")
    
    # Get effect radius if applicable
    radius = item.find("Effect.EffectTargetRadius")
    if radius and radius():
        print(f"Effect Radius: {radius()}")

Item: Zorsines, Sarmatian Swordshaper

Effect Scope (literal): Radius
Localized: in range:


## Find Items by Effect Scope

Different scopes:
- `Local` - Only the building the item is slotted in
- `Radius` - Buildings within a certain radius
- `Area` - All buildings in an area
- `Session` - All buildings in the session
- `Island` - All buildings on the island

In [11]:
def find_items_by_scope(scope="Radius", max_results=10):
    """Find items with specific effect scope."""
    items_template = assets.templates["Item"]
    results = []
    
    for item in items_template.assets:
        if not hasattr(item, 'Effect'):
            continue
        
        item_scope = item.Effect.EffectScope()
        if item_scope == scope:
            # Get radius if applicable
            radius_attr = item.find("Effect.EffectTargetRadius")
            radius = radius_attr() if radius_attr else None
            
            results.append((item.text(), radius, item.guid))
        
        if len(results) >= max_results:
            break
    
    return results

# Find items with radius effects
radius_items = find_items_by_scope("Radius", max_results=10)

print("Items with Radius Effects:\n")
for name, radius, guid in radius_items:
    radius_str = f"(radius: {radius})" if radius else ""
    print(f"  • {name:45s} {radius_str}")

Items with Radius Effects:

  • Gaius Julius Lupus, Castor of Fortunes        
  • Brutus Julius Lupus, Pollux of Polities       
  • Abdfil, Elephant Handler                      
  • Elephant Handler                              
  • Aodhan, Master Lutist of the Scathach         
  • Abuccus                                       
  • Servia Bellia, Lily of the Coast              
  • Publius Quintus, Amphipraetorian              
  • Virtuous Volunteer                            
  • Apion Mochthos Apicius, of Apeiron Appetite   


## Explore AreaBuff Template

AreaBuff assets define area-wide effects (not item-based):

In [12]:
# Check if AreaBuff template exists
if "AreaBuff" in assets.templates.elements:
    area_buff_template = assets.templates["AreaBuff"]
    print(f"Found {len(area_buff_template.assets)} AreaBuff assets\n")
    
    # Show first 10
    print("First 10 AreaBuff assets:")
    for i, buff in enumerate(list(area_buff_template.assets)[:10], 1):
        name = buff.text()
        print(f"{i:2d}. {name} (GUID: {buff.guid})")
else:
    print("AreaBuff template not found")

Found 235 AreaBuff assets

First 10 AreaBuff assets:


TypeError: 'NoneType' object is not callable

## Extract Area Buff Properties

AreaBuff assets have different properties than BuildingBuff:

In [ ]:
if "AreaBuff" in assets.templates.elements:
    # Get first area buff
    area_buff = list(assets.templates["AreaBuff"].assets)[0]
    name = area_buff.text()
    
    print(f"AreaBuff: {name}\n")
    
    # Use buff_ui to get all effects
    buff_list = area_buff.buff_ui
    
    if buff_list:
        print("Effects:")
        for buff_ui in buff_list:
            print(f"  • {buff_ui}")
    else:
        print("No effects found (might be in different structure)")
    
    # Check for specific area buff properties
    if hasattr(area_buff, 'AreaBuff'):
        print("\nAreaBuff Properties:")
        area_buff.AreaBuff.print_tree()

## Example: RadiusEffectRangeUpgrade

Some area buffs increase the effect range of buildings:

In [ ]:
def find_radius_range_buffs(max_results=10):
    """Find AreaBuff assets that modify effect radius."""
    if "AreaBuff" not in assets.templates.elements:
        return []
    
    results = []
    area_buff_template = assets.templates["AreaBuff"]
    
    for buff in area_buff_template.assets:
        # Use buff_ui and check for radius-related effects
        buff_ui_list = buff.buff_ui
        has_radius_effect = False
        
        for buff_ui in buff_ui_list:
            buff_text = buff_ui.text()
            if 'radius' in buff_text.lower() or 'range' in buff_text.lower():
                has_radius_effect = True
                break
        
        if has_radius_effect:
            results.append((buff.text(), buff_ui_list, buff.guid))
        
        if len(results) >= max_results:
            break
    
    return results

# Find buffs that modify effect radius
radius_buffs = find_radius_range_buffs(max_results=5)

if radius_buffs:
    print("AreaBuffs that modify Effect Radius:\n")
    for name, buff_ui_list, guid in radius_buffs:
        print(f"  {name}")
        for buff_ui in buff_ui_list:
            print(f"    → {buff_ui}")
        print()
else:
    print("No radius effect buffs found")

## Compare Effect Scopes Distribution

See which effect scopes are most common:

In [ ]:
from collections import Counter

scope_distribution = Counter()
items_template = assets.templates["Item"]

for item in list(items_template.assets)[:200]:  # Sample first 200 items
    if hasattr(item, 'Effect'):
        scope = item.Effect.EffectScope()
        if scope:
            scope_distribution[scope] += 1

print("Effect Scope Distribution (from 200 items):\n")

for scope, count in scope_distribution.most_common():
    # Get localized text using ui_text
    scope_text = scope
    
    # Try to get UI text from first item with this scope
    for item in items_template.assets:
        if hasattr(item, 'Effect') and item.Effect.EffectScope() == scope:
            scope_attr = item.Effect.EffectScope
            if hasattr(scope_attr, 'ui_text') and scope_attr.ui_text:
                scope_text = scope_attr.ui_text()
            break
    
    print(f"  {scope_text:20s} ({scope:15s}): {count:3d} items")

## Find Largest Effect Radius

Find items with the largest effect radius:

In [ ]:
radius_items = []
items_template = assets.templates["Item"]

for item in items_template.assets:
    if not hasattr(item, 'Effect'):
        continue
    
    radius_attr = item.find("Effect.EffectTargetRadius")
    if radius_attr and radius_attr():
        radius = radius_attr()
        radius_items.append((item.text(), radius, item.guid))

# Sort by radius (descending)
radius_items.sort(key=lambda x: x[1], reverse=True)

print("Items with Largest Effect Radius:\n")
for i, (name, radius, guid) in enumerate(radius_items[:10], 1):
    print(f"{i:2d}. {name:45s} Radius: {radius}")

## Analyze Area Effect Combinations

Find items with both radius effects and specific buff types:

In [ ]:
# Find radius items with productivity buffs
radius_productivity = []
items_template = assets.templates["Item"]

for item in list(items_template.assets)[:100]:  # Sample first 100
    if not hasattr(item, 'Effect'):
        continue
    
    # Check for radius scope
    scope = item.Effect.EffectScope()
    if scope != "Radius":
        continue
    
    # Check for productivity in buffs
    has_productivity = False
    
    for buff_ui in item.buff_ui:
        buff_text = buff_ui.text()
        if 'productivity' in buff_text.lower():
            has_productivity = True
            break
    
    if has_productivity:
        radius_attr = item.find("Effect.EffectTargetRadius")
        radius = radius_attr() if radius_attr else None
        radius_productivity.append((item.text(), radius))

if radius_productivity:
    print("Radius Items with Productivity Buffs:\n")
    for name, radius in radius_productivity:
        radius_str = f"(radius: {radius})" if radius else ""
        print(f"  • {name:45s} {radius_str}")
else:
    print("No radius items with productivity buffs found in sample")

## Next Steps

- `06_backtrack_effects.ipynb` - Find items by buff type
- `08_localize_literals.ipynb` - Localize scope literals and other datasets